# assignment in short
Rows is users, columns is movies, entries is 0/1 --> large sparse binary matrix.

LSH(locality sensitive hashing) is a technique for quickly finding similar items in large datasets without comparing everything. Similar items are likely to get the same hash value. LSH uses special hash functions that cluster similar things together.
Each user is represented as the set of movies that they rated. minhash signature is its compressed representation.
Signatures are split up into 'bands', each band is split up into rows. If two users have identical rows in at least one band, they might be similar users.

for these 'might be similar' users, compute jaccard similarity to find pairs of most similar users.

In [14]:
# load user_movie_rating.npy and parse columns

import numpy as np
import os

# check if file exists
user_movie_rating_path = "user_movie_rating.npy"
if not os.path.exists(user_movie_rating_path):
    raise FileNotFoundError(f"{user_movie_rating_path} not found in current directory")

# load data and raise error if data is not as expected
data = np.load(user_movie_rating_path, mmap_mode='r') # mmap_mode='r' for large files
if data.ndim != 2 or data.shape[1] != 3:
    raise ValueError("Expected a 2D array with 3 columns: user_id, movie_id, rating")

# parse columns
users = data[:, 0].astype(int)
movies = data[:, 1].astype(int)
ratings = data[:, 2].astype(int)

#! at the end the specific rating does not matter for building the user-item matrix

n_users = users.max()
n_movies = movies.max()

print(f"Loaded {data.shape[0]} interactions")
print(f"Users: {users.max()} ")
print(f"Movies: {movies.max()} ")

# example: show first 10 raw rows and their mapped indices
print("First 10 raw rows (user_id, movie_id, rating):")
print(data[:10])



Loaded 65225506 interactions
Users: 103703 
Movies: 17770 
First 10 raw rows (user_id, movie_id, rating):
[[  1  30   3]
 [  1 157   3]
 [  1 173   4]
 [  1 175   5]
 [  1 191   2]
 [  1 197   3]
 [  1 241   3]
 [  1 295   4]
 [  1 299   3]
 [  1 329   4]]


# what sparse matrix storage scheme to use
| Format | Matrix × Vector | Get Item | Fancy Get | Set Item | Fancy Set | Solvers | Notes |
|--------|------------------|----------|-----------|----------|-----------|---------|--------|
| **DIA** | sparsetools | . | . | . | . | iterative | has data array, specialized |
| **LIL** | via CSR | yes | yes | yes | yes | iterative | arithmetics via CSR, incremental construction |
| **DOK** | python | yes | one axis only | yes | yes | iterative | O(1) item access, incremental construction |
| **COO** | sparsetools | . | . | . | . | iterative | has data array, facilitates fast conversion |
| **CSR** | sparsetools | yes | yes | slow | . | any | has data array, fast row-wise ops |
| **CSC** | sparsetools | yes | yes | slow | . | any | has data array, fast column-wise ops |
| **BSR** | sparsetools | . | . | . | . | specialized | has data array, specialized |

For LSH we want fast row access because we create hash values from users(set of movies they rated) CSR is good for this
For the cadidates to be similar, we need to calculate jaccard similarity. we need to look at the intersections of the two movie sets of the users. 

To build the sparse matrix COO LIL and DOK are mostly used. 


In [ ]:
# create a sparse user-item rating matrix

from scipy.sparse import coo_matrix

#scipy uses ID's starting from 0
users -= 1
movies -= 1

data_values = np.ones_like(users, dtype=np.uint8) # rating presence indicator, ratings is ignored
coo = coo_matrix((data_values, (users, movies)), shape=(n_users, n_movies), dtype=bool)
print(f"Sparse rating matrix shape: {coo.shape}, nnz={coo.nnz}")
del users, movies, ratings, data
csr = coo.tocsr()
del coo

#print some csr data
print(f"CSR matrix shape: {csr.shape}, nnz={csr.nnz}")
print(f"CSR matrix data sample (first 10 entries): {csr.data[:40]}")

Sparse rating matrix shape: (103703, 17770), nnz=65225506
CSR matrix shape: (103703, 17770), nnz=65225506
CSR matrix data sample (first 10 entries): [ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True]
